# Itemset — Julia setup on Colab

Python kernel; Julia runs as subprocesses. This starter verifies the environment, tests, and transaction input. Eclat, benchmarks, and both applications remain to be implemented. No fixture is presented as a mined result.

Local repository: export a committed ZIP with `git archive --format=zip --output=../Itemset-colab.zip HEAD`, upload this notebook to Colab, and upload the ZIP when prompted. For a future public GitHub repository, set its URL and exact commit below. The bootstrap downloads Julia 1.13.2 from the official server and checks its SHA-256.

In [ ]:
from pathlib import Path
import os
import subprocess
import zipfile

REPO_URL = ""  # Optional future public GitHub HTTPS URL.
REVISION = ""  # Required exact commit if REPO_URL is set; record archive commit too.
ROOT = Path("/content/Itemset")

if not (ROOT / "Project.toml").exists():
    if REPO_URL:
        if not REVISION:
            raise ValueError("Set an exact commit in REVISION for reproducibility")
        subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
        subprocess.run(["git", "-C", str(ROOT), "checkout", "--detach", REVISION], check=True)
    else:
        from google.colab import files
        print("Upload Itemset-colab.zip exported from a local commit")
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("Upload exactly one repository ZIP")
        ROOT.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(next(iter(uploaded))) as archive:
            for member in archive.infolist():
                if not (ROOT / member.filename).resolve().is_relative_to(ROOT.resolve()):
                    raise ValueError("ZIP member is outside the repository")
            archive.extractall(ROOT)

if not (ROOT / "Project.toml").is_file():
    raise RuntimeError("Repository root must contain Project.toml")
print("Repository:", ROOT)
print("Revision:", REVISION or "local archive; record its commit before final experiments")


In [ ]:
import hashlib
import platform
import urllib.request

if platform.system() != "Linux" or platform.machine() not in ("x86_64", "AMD64"):
    raise RuntimeError("This bootstrap targets the standard Linux x86_64 Colab runtime")
JULIA_VERSION = "1.13.2"
URL = "https://julialang-s3.julialang.org/bin/linux/x64/1.13/julia-1.13.2-linux-x86_64.tar.gz"
EXPECTED_SHA256 = "2ecdfc9890ab0a565fdd92404c915ba2a04a933dbd5d292cc224d13819d17d07"
JULIA_HOME = Path("/content/julia-1.13.2")
JULIA = JULIA_HOME / "bin" / "julia"
if not JULIA.exists():
    archive = Path("/content/julia-1.13.2.tar.gz")
    urllib.request.urlretrieve(URL, archive)
    with archive.open("rb") as stream:
        actual = hashlib.file_digest(stream, "sha256").hexdigest()
    if actual != EXPECTED_SHA256:
        raise RuntimeError("Julia archive SHA-256 mismatch")
    JULIA_HOME.mkdir(parents=True, exist_ok=True)
    subprocess.run(["tar", "-xzf", str(archive), "-C", str(JULIA_HOME),
                    "--strip-components=1"], check=True)
ENV = dict(os.environ, JULIA_NUM_THREADS="1", JULIA_DEPOT_PATH="/content/itemset-julia-depot")

def run_julia(*args):
    return subprocess.run([str(JULIA), "--startup-file=no", "--project=.", *args],
                          cwd=ROOT, env=ENV, check=True)

run_julia("-e", 'VERSION == v"1.13.2" || error("Unexpected Julia version"); using Pkg; Pkg.instantiate()')


In [ ]:
run_julia("scripts/check_setup.jl")
run_julia("test/runtests.jl")
run_julia("demo.jl", "--check-input", "data/toy/baskets.txt")


## Hung — Eclat, tests, scalability, output size

Add the paper-based baseline, hand trace, version selection, and real Eclat correctness checks (DAT-8/29/30). Release only after DAT-13 independent SPMF validation. Later add nested scalability subsets and output-size plots. The setup tests above cover infrastructure, not miner correctness.

## Thien — Market baskets, runtime, transaction width

Add public labelled retail data acquisition, checksum/license, cleaning, reversible mappings, own Eclat invocation, rule tests, top-10 lift rules and interpretation (DAT-10/14/37). Add the common benchmark runner and runtime/width plots. Pin approved dependencies and record seed 20261018.

## Phat — Image patterns, reference correctness, memory

Add real image data, descriptors, fixed-seed codebook, transactions, visual-word patches, own Eclat mining, supporting images, and interpretation (DAT-11/15/38). Add independent SPMF comparisons and peak RSS commands. Integrate all three owner-tested modules for DAT-25.

## Clean-run checklist

Restart the runtime and run all cells after modules are implemented. Record revision, dataset hashes, dependencies, parameters, stdout, and failures. Publish a public Colab link in README only after verifying access and a complete clean run. Local setup validation does not prove this Colab runtime has been exercised.